In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
import bs4
import requests
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from dotenv import load_dotenv, dotenv_values

load_dotenv()
env_dict = dotenv_values(".env")
print("--- Contents of .env file ---")
for key, value in env_dict.items():
    # Masking the value so you don't accidentally print your full API key to your screen!
    masked_value = f"{value[:5]}...{value[-4:]}" if len(value) > 10 else value
    print(f"{key}: {masked_value}")

--- Contents of .env file ---
OPENAI_API_KEY: sk-pr...-PAA
GOOGLE_API_KEY: AIzaS...3M6o
TAVILY_API_KEY: tvly-...keLk
USER_AGENT: rag-f.../1.0
LANGSMITH_TRACING: true
LANGSMITH_ENDPOINT: https....com
LANGSMITH_API_KEY: lsv2_...2276
LANGSMITH_PROJECT: langc...demy
LANGCHAIN_TRACING_V2: true
LANGCHAIN_API_KEY: lsv2_...13cd
LANGCHAIN_PROJECT: langc...demo


In [11]:
# %%
import os

cwd = os.getcwd()
print(f"Kernel is currently running in: \n{cwd}\n")

print("Files in this directory:")
for file in os.listdir()[:10]: # Printing the first 10 files
    print(f" - {file}")

Kernel is currently running in: 
/Users/isaacfrith/rag-from-scratch

Files in this directory:
 - rag-from-scratch
 - rag.ipynb


In [2]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [7]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = InMemoryVectorStore(embeddings)


In [10]:
def load_web_page(url: str, bs_kwargs: dict | None = None) -> list[Document]:
    response = requests.get(url)
    response.raise_for_status
    soup = bs4.BeautifulSoup(response.text, "html.parser", **(bs_kwargs or {}))
    return [Document(page_content=soup.get_text(), metadata={"source": url})]

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
docs = load_web_page(
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    bs_kwargs={"parse_only": bs4_strainer},
)

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 43047


In [11]:
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In
